# EMG — Line In Left: bandpass reference + powerline filters

Loads **only** `emg_line_L.csv` from a recording folder you pick.

**Pipeline**

1. Raw signal
2. → **Bandpass 5–500 Hz** (Butterworth, zero-phase `sosfiltfilt`) — this is the
   **reference signal** everything downstream is built on and compared against
3. → three powerline filters applied *to the reference*, each independently:
   - **Feed-forward comb** — `y[n] = x[n] − x[n−M]`, `M = round(fs/f0)`
   - **IIR comb, 5th order, normalized** — `[(1 − z⁻ᴹ)/(1 − r·z⁻ᴹ)]⁵`
   - **IIR notch, 5th order** — `[iirnotch(k·f0, Q)]⁵` at every mains harmonic

**Figures** (all interactive Plotly)

| | |
|---|---|
| **Fig 1** | Raw vs. reference — time domain, linked x-axes |
| **Fig 2** | Raw vs. reference — amplitude spectrum |
| **Fig 3** | The 3 filter outputs vs. reference — time domain |
| **Fig 4** | The 3 filter outputs vs. reference — amplitude spectrum |

Run the cells top to bottom, then use the widget in the last cell.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal as sig

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, clear_output

RECORDINGS_DIR = Path("recordings")
CHANNEL_FILE = "emg_line_L.csv"   # this notebook uses this file and nothing else

# Powerline-filter constants — same values as comb_filter_verification.py /
# notch_filter_verification.py, so results here line up with those studies.
F0            = 49.97328   # Hz — measured mains frequency
R_M           = 0.9        # IIR comb feedback coefficient
Q             = 30.0       # IIR notch quality factor (~1.7 Hz wide at 50 Hz)
HARMONICS_MAX = 500.0      # Hz — notch every harmonic up to here

In [ ]:
# ── folder discovery ──────────────────────────────────────────────────────────

def find_folders(root: Path = RECORDINGS_DIR):
    """Every folder under *root* that contains an emg_line_L.csv.

    Searched recursively, so nested collections such as recordings/extras/…
    show up too. Returns paths relative to *root*, sorted.
    """
    if not root.exists():
        return []
    return sorted(p.parent.relative_to(root).as_posix()
                  for p in root.rglob(CHANNEL_FILE))


# ── data ──────────────────────────────────────────────────────────────────────

def load_line_L(folder: str):
    """Read emg_line_L.csv from *folder* → (time, signal, fs)."""
    path = RECORDINGS_DIR / folder / CHANNEL_FILE
    if not path.exists():
        raise FileNotFoundError(f"{CHANNEL_FILE} not found in {path.parent}")
    df = pd.read_csv(path)
    t = df["time"].to_numpy(dtype=float)
    x = df["data"].to_numpy(dtype=float)
    # median of the sample-to-sample deltas — immune to a single glitched timestamp
    fs = 1.0 / float(np.median(np.diff(t)))
    return t, x, fs


# ── reference signal ──────────────────────────────────────────────────────────

def bandpass_filter(x: np.ndarray, fs: float,
                    low_hz: float = 5.0, high_hz: float = 500.0,
                    order: int = 4) -> np.ndarray:
    """Zero-phase Butterworth bandpass, SOS form for numerical stability.

    5 Hz at fs = 44.1 kHz is a very low normalised frequency; transfer-function
    (b, a) coefficients lose precision there, second-order sections do not.
    """
    sos = sig.butter(order, [low_hz, high_hz], btype="band", fs=fs, output="sos")
    return sig.sosfiltfilt(sos, x)


# ── FFT ───────────────────────────────────────────────────────────────────────

def rfft_single_sided(x: np.ndarray, fs: float):
    """Single-sided amplitude spectrum → (freqs, magnitude)."""
    N = len(x)
    freqs = np.fft.rfftfreq(N, d=1.0 / fs)
    mag = np.abs(np.fft.rfft(x)) * (2.0 / N)
    mag[0] /= 2.0                      # DC is not mirrored
    if N % 2 == 0:
        mag[-1] /= 2.0                 # nor is Nyquist, when it exists
    return freqs, mag


def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(x ** 2)))


def peak_near(fr, mag, f_target, f_bw=1.5):
    """Largest spectral magnitude within ±f_bw Hz of f_target."""
    m = (fr >= f_target - f_bw) & (fr <= f_target + f_bw)
    return float(np.max(mag[m])) if m.any() else np.nan

In [ ]:
# ── powerline filters, applied to the bandpassed reference ───────────────────
#
# "Order N" means N cascaded identical sections, the same definition used in
# comb_filter_verification.py and notch_filter_verification.py.
#
# All three are applied ZERO-PHASE, by running one section's filtfilt N times
# rather than expanding the cascade into a single high-degree polynomial.
# The two are equivalent in magnitude — filtfilt gives |H|² per pass, so N
# passes give |H|^2N = |H^N|² — but the repeated-section form is much better
# conditioned and roughly 4x faster. Expanding an order-5 comb would produce a
# degree-4410 polynomial whose N-fold poles sit essentially on the unit circle.


def comb_section(fs: float, kind: str, f0: float = F0,
                 r_m: float = R_M, normalize: bool = True):
    """(b, a) for ONE comb section. kind is "ff" (feed-forward) or "iir".

    Feed-forward:  H(z) = 1 − z⁻ᴹ                  passband peak 2
    IIR (feedback):H(z) = (1 − z⁻ᴹ)/(1 − r·z⁻ᴹ)    passband peak 2/(1+r)

    Both arch ABOVE unity between the harmonics, and filtfilt squares that, so
    the raw designs amplify everything that is not mains (feed-forward by ×4).
    normalize=True scales the numerator so one section peaks at exactly 1.0 —
    notch depth and position are untouched, only the passband boost goes away.
    """
    M = int(round(fs / f0))
    b = np.zeros(M + 1)
    b[0], b[M] = 1.0, -1.0

    if kind == "ff":
        a = np.array([1.0])
        if normalize:
            b = b * 0.5
    else:
        a = np.zeros(M + 1)
        a[0], a[M] = 1.0, -r_m
        if normalize:
            b = b * (1.0 + r_m) / 2.0
    return b, a, M


def comb_filter(x, fs, kind, order=1, f0=F0, r_m=R_M, normalize=True):
    """Apply a comb of the given cascade order, zero-phase."""
    b, a, _ = comb_section(fs, kind, f0, r_m, normalize)
    y = x.astype(float)
    for _ in range(order):
        y = sig.filtfilt(b, a, y)
    return y


def iir_notch_filter(x, fs, order=5, f0=F0, q=Q, harmonics_max=HARMONICS_MAX):
    """Cascade of order-N IIR notches at every harmonic of f0, zero-phase.

    The order-N cascade is built as tiled second-order sections and run through
    sosfiltfilt. Never expand it into a degree-2N transfer function: the N-fold
    poles land on |z| = 1 and a direct-form evaluation blows up.

    No normalization variant here — each biquad notch already has a passband
    gain of exactly 1, so unlike the comb it cannot inflate the signal.
    """
    y = x.astype(float)
    k = 1
    while k * f0 <= harmonics_max and k * f0 < fs / 2:
        b1, a1 = sig.iirnotch(k * f0, q, fs=fs)
        sos = np.tile(np.concatenate([b1, a1]), (order, 1))
        y = sig.sosfiltfilt(sos, y)
        k += 1
    return y

In [ ]:
# ── plotting helpers ──────────────────────────────────────────────────────────

RAW_COLOR = "#1565C0"
REF_COLOR = "#E65100"
BP_GREY   = "#777777"

# one colour per powerline filter, matching the families used in the *_verification scripts
FILTER_COLORS = {
    "ff_comb":   "#2E7D32",   # green  — feed-forward comb
    "iir_comb":  "#8E0000",   # red    — IIR comb
    "iir_notch": "#6A1B9A",   # purple — IIR notch
}


def envelope(t, x, n_bins=6000):
    """Peak-preserving decimation for plotting long signals.

    Splits the signal into n_bins time-bins and keeps the ACTUAL min and max
    sample of each bin. Unlike plain striding this can never hide a spike, so
    the on-screen amplitude stays truthful while the point count drops to
    ~2·n_bins — which is what keeps pan/zoom responsive at 200k+ samples.
    """
    n = len(x)
    if n <= 2 * n_bins:
        return t, x
    bin_len = n // n_bins
    usable = bin_len * n_bins
    idx = np.arange(usable).reshape(n_bins, bin_len)
    xb = x[:usable].reshape(n_bins, bin_len)
    rows = np.arange(n_bins)
    keep = np.union1d(idx[rows, np.argmin(xb, axis=1)],
                      idx[rows, np.argmax(xb, axis=1)])
    if usable < n:                       # keep the leftover tail
        keep = np.union1d(keep, np.arange(usable, n))
    return t[keep], x[keep]

In [ ]:
# ── main analysis ─────────────────────────────────────────────────────────────

def analyze(folder: str,
            low_hz: float = 5.0,
            high_hz: float = 500.0,
            bp_order: int = 4,
            fft_max_hz: float = 600.0,
            log_y: bool = False,
            f0: float = F0,
            ff_order: int = 1,
            comb_order: int = 5,
            notch_order: int = 5,
            normalize_combs: bool = True,
            harmonics_max: float = HARMONICS_MAX):
    """Bandpass emg_line_L.csv from *folder*, then apply the 3 powerline filters."""

    t, raw, fs = load_line_L(folder)
    print(f"{folder}/{CHANNEL_FILE}")
    print(f"  fs = {fs:,.1f} Hz   {len(raw):,} samples   {t[-1] - t[0]:.3f} s")

    nyq = fs / 2.0
    if high_hz >= nyq:
        high_hz = nyq * 0.99
        print(f"  ! high cutoff clipped to {high_hz:.1f} Hz (Nyquist = {nyq:,.0f} Hz)")

    # ── reference ─────────────────────────────────────────────────────────────
    ref = bandpass_filter(raw, fs, low_hz, high_hz, bp_order)
    ref_label = f"Reference — bandpass {low_hz:g}–{high_hz:g} Hz"
    M = int(round(fs / f0))
    print(f"  {ref_label} (Butterworth order {bp_order}, zero-phase)")
    print(f"  f0 = {f0:.5f} Hz   M = round(fs/f0) = {M}   "
          f"r = {R_M}   Q = {Q}   normalize combs = {normalize_combs}")

    # ── the three powerline filters, each applied to the reference ────────────
    norm_tag = "normalized" if normalize_combs else "un-normalized"
    outputs = [
        ("ff_comb",
         f"Feed-forward comb, order {ff_order} ({norm_tag})",
         comb_filter(ref, fs, "ff", ff_order, f0, R_M, normalize_combs)),
        ("iir_comb",
         f"IIR comb, order {comb_order} ({norm_tag}), r={R_M}",
         comb_filter(ref, fs, "iir", comb_order, f0, R_M, normalize_combs)),
        ("iir_notch",
         f"IIR notch, order {notch_order}, Q={Q:g}",
         iir_notch_filter(ref, fs, notch_order, f0, Q, harmonics_max)),
    ]

    # ── spectra ───────────────────────────────────────────────────────────────
    fr, fft_raw = rfft_single_sided(raw, fs)
    _,  fft_ref = rfft_single_sided(ref, fs)
    specs = {key: rfft_single_sided(y, fs)[1] for key, _, y in outputs}
    fm = fr <= fft_max_hz

    # ── numeric summary vs the reference ──────────────────────────────────────
    rms_ref = rms(ref)
    ref_f0, ref_2f0 = peak_near(fr, fft_ref, f0), peak_near(fr, fft_ref, 2 * f0)
    print(f"\n  RMS reference = {rms_ref:.6f} V")
    print(f"  {'filter':<44}{'RMS (V)':>11}{'x ref':>8}"
          f"{'att@f0':>9}{'att@2f0':>9}")
    print("  " + "-" * 81)
    for key, label, y in outputs:
        mag = specs[key]
        a1 = 20 * np.log10(ref_f0 / (peak_near(fr, mag, f0) + 1e-20))
        a2 = 20 * np.log10(ref_2f0 / (peak_near(fr, mag, 2 * f0) + 1e-20))
        print(f"  {label:<44}{rms(y):>11.6f}{rms(y)/rms_ref:>8.3f}"
              f"{a1:>+8.1f}dB{a2:>+8.1f}dB")

    # ── Fig 1: raw vs reference, time ─────────────────────────────────────────
    t_r, y_r = envelope(t, raw)
    t_b, y_b = envelope(t, ref)
    fig1 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                         subplot_titles=["Raw", ref_label])
    fig1.add_trace(go.Scattergl(x=t_r, y=y_r, name="Raw",
                                line=dict(width=0.7, color=RAW_COLOR)), row=1, col=1)
    fig1.add_trace(go.Scattergl(x=t_b, y=y_b, name="Reference (bandpassed)",
                                line=dict(width=0.7, color=REF_COLOR)), row=2, col=1)
    fig1.update_yaxes(title_text="Amplitude (V)", row=1, col=1)
    fig1.update_yaxes(title_text="Amplitude (V)", row=2, col=1)
    fig1.update_xaxes(title_text="Time (s)", row=2, col=1)
    fig1.update_layout(title_text=f"Fig 1 — {folder} — raw vs reference (time)",
                       height=600, template="plotly_white", hovermode="x unified",
                       legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"))
    fig1.show()

    # ── Fig 2: raw vs reference, spectrum ─────────────────────────────────────
    fig2 = go.Figure()
    fig2.add_trace(go.Scattergl(x=fr[fm], y=fft_raw[fm], name="Raw",
                                line=dict(width=1.0, color=RAW_COLOR)))
    fig2.add_trace(go.Scattergl(x=fr[fm], y=fft_ref[fm], name="Reference (bandpassed)",
                                line=dict(width=1.2, color=REF_COLOR)))
    for fc in (low_hz, high_hz):
        if fc <= fft_max_hz:
            fig2.add_vline(x=fc, line=dict(color="rgba(0,0,0,0.35)", width=1, dash="dash"),
                           annotation_text=f"{fc:g} Hz", annotation_position="top",
                           annotation_font_size=9)
    fig2.update_layout(title_text=f"Fig 2 — {folder} — raw vs reference (spectrum)",
                       xaxis_title="Frequency (Hz)", yaxis_title="Magnitude (V)",
                       yaxis_type="log" if log_y else "linear",
                       height=440, template="plotly_white", hovermode="x unified",
                       legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"))
    fig2.show()

    # ── Fig 3: the 3 filter outputs vs reference, time ────────────────────────
    fig3 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                         subplot_titles=[lbl for _, lbl, _ in outputs])
    for r, (key, label, y) in enumerate(outputs, 1):
        # reference drawn underneath in grey so each panel is directly comparable
        fig3.add_trace(go.Scattergl(
            x=t_b, y=y_b, name="Reference", legendgroup="ref",
            line=dict(width=0.7, color=BP_GREY), opacity=0.55,
            showlegend=(r == 1)), row=r, col=1)
        t_f, y_f = envelope(t, y)
        fig3.add_trace(go.Scattergl(
            x=t_f, y=y_f, name=f"{label}  (×{rms(y)/rms_ref:.2f} RMS)",
            legendgroup=key, line=dict(width=0.8, color=FILTER_COLORS[key])),
            row=r, col=1)
        fig3.update_yaxes(title_text="Amplitude (V)", row=r, col=1)
    fig3.update_xaxes(title_text="Time (s)", row=3, col=1)
    fig3.update_layout(
        title_text=(f"Fig 3 — {folder} — powerline filters applied to the reference (time)"
                    "<br><sup>grey = bandpass reference in every panel; "
                    "legend shows RMS relative to it</sup>"),
        height=860, template="plotly_white", hovermode="x unified",
        legend=dict(orientation="h", y=1.06, x=0.5, xanchor="center"))
    fig3.show()

    # ── Fig 4: the 3 filter outputs vs reference, spectrum ────────────────────
    fig4 = go.Figure()
    fig4.add_trace(go.Scattergl(x=fr[fm], y=fft_ref[fm], name="Reference (bandpassed)",
                                line=dict(width=1.0, color=BP_GREY)))
    for key, label, y in outputs:
        fig4.add_trace(go.Scattergl(x=fr[fm], y=specs[key][fm], name=label,
                                    line=dict(width=1.2, color=FILTER_COLORS[key])))
    for k in range(1, int(min(fft_max_hz, harmonics_max) / f0) + 1):
        fig4.add_vline(x=k * f0, line=dict(color="rgba(180,0,0,0.30)", width=1, dash="dot"))
    fig4.update_layout(
        title_text=(f"Fig 4 — {folder} — powerline filters applied to the reference (spectrum)"
                    f"<br><sup>red dotted = harmonics of f0 = {f0:.3f} Hz</sup>"),
        xaxis_title="Frequency (Hz)", yaxis_title="Magnitude (V)",
        yaxis_type="log" if log_y else "linear",
        height=520, template="plotly_white", hovermode="x unified",
        legend=dict(orientation="h", y=1.10, x=0.5, xanchor="center"))
    fig4.show()

    print("Done.")

In [ ]:
clear_output(wait=True)

folders = find_folders()

if not folders:
    print(f"No folder under '{RECORDINGS_DIR}/' contains {CHANNEL_FILE}.")
else:
    def _num(w_cls, value, desc, width="230px", **kw):
        return w_cls(value=value, description=desc, style={"description_width": "initial"},
                     layout=widgets.Layout(width=width), **kw)

    _w_folder = widgets.Dropdown(
        options=folders, value=folders[0], description="Recording:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="520px"))

    # reference (bandpass) controls
    _w_low   = _num(widgets.BoundedFloatText, 5.0,   "Low cutoff (Hz):", "230px",
                    min=0.1, max=2000.0, step=1.0)
    _w_high  = _num(widgets.BoundedFloatText, 500.0, "High cutoff (Hz):", "230px",
                    min=1.0, max=20000.0, step=10.0)
    _w_bpord = _num(widgets.BoundedIntText,   4,     "BP order:", "160px",
                    min=1, max=8, step=1)

    # powerline-filter controls
    _w_f0    = _num(widgets.BoundedFloatText, F0,    "Mains f0 (Hz):", "230px",
                    min=40.0, max=70.0, step=0.001)
    _w_ffo   = _num(widgets.BoundedIntText,   1,     "FF comb order:", "200px",
                    min=1, max=10, step=1)
    _w_ico   = _num(widgets.BoundedIntText,   5,     "IIR comb order:", "200px",
                    min=1, max=10, step=1)
    _w_ino   = _num(widgets.BoundedIntText,   5,     "IIR notch order:", "200px",
                    min=1, max=10, step=1)
    _w_hmax  = _num(widgets.BoundedFloatText, HARMONICS_MAX, "Notch harmonics to (Hz):",
                    "260px", min=50.0, max=20000.0, step=50.0)
    _w_norm  = widgets.Checkbox(
        value=True, description="Normalize combs (passband peak → 1)", indent=False,
        layout=widgets.Layout(width="330px"))

    # display controls
    _w_fft   = _num(widgets.BoundedFloatText, 600.0, "FFT max (Hz):", "220px",
                    min=10.0, max=24000.0, step=50.0)
    _w_log   = widgets.Checkbox(value=False, description="Log magnitude axis",
                                indent=False, layout=widgets.Layout(width="200px"))

    _w_run = widgets.Button(description="▶  Filter & Plot", button_style="primary",
                            layout=widgets.Layout(width="190px", height="38px"))
    _w_out = widgets.Output()

    def _on_run(b):
        with _w_out:
            clear_output(wait=True)
            try:
                analyze(
                    _w_folder.value,
                    low_hz=float(_w_low.value), high_hz=float(_w_high.value),
                    bp_order=int(_w_bpord.value),
                    fft_max_hz=float(_w_fft.value), log_y=bool(_w_log.value),
                    f0=float(_w_f0.value),
                    ff_order=int(_w_ffo.value), comb_order=int(_w_ico.value),
                    notch_order=int(_w_ino.value),
                    normalize_combs=bool(_w_norm.value),
                    harmonics_max=float(_w_hmax.value),
                )
            except Exception as exc:
                print(f"{type(exc).__name__}: {exc}")

    _w_run.on_click(_on_run)

    def _section(title, *rows):
        return widgets.VBox([widgets.HTML(f"<b>{title}</b>"), *rows])

    display(widgets.VBox([
        widgets.HTML(f"<b>Pick a recording folder — only <code>{CHANNEL_FILE}</code> "
                     "is read.</b>"),
        _w_folder,
        _section("Reference signal (bandpass)",
                 widgets.HBox([_w_low, _w_high, _w_bpord])),
        _section("Powerline filters (applied to the reference)",
                 widgets.HBox([_w_f0, _w_ffo, _w_ico]),
                 widgets.HBox([_w_ino, _w_hmax, _w_norm])),
        _section("Display",
                 widgets.HBox([_w_fft, _w_log])),
        _w_run,
        _w_out,
    ]))